# timeseries-qc quickstart

Five minutes, one synthetic SCADA feed, four flavors of bad data.

This notebook builds a small **boiler-plant** style dataset (flow, pressure, temperature),
deliberately breaks it in the ways real historian data breaks, then runs it through
`timeseries-qc` end-to-end:

1. Generate synthetic tag data with injected **range violations**, a **flatline**, **null
   values**, and a **timestamp gap**
2. Define thresholds in a **YAML rule config** (no Python required for this part)
3. Run `tsqc.check()` to classify every row `good` / `sus` / `bad`
4. Inspect the results: `summary()`, `issue_summary()`, `check_timestamps()`
5. Render the **Plotly quality timeline**
6. Export a self-contained HTML report

If you just want to see the payoff, skip to [Step 5](#5.-Visualize:-the-quality-timeline).

> Install: `pip install timeseries-qc` (also installs pandas + plotly as dependencies).


In [ ]:
import numpy as np
import pandas as pd
import tsqc

print("tsqc", tsqc.__version__)


## 1. Generate synthetic SCADA data

Three tags, one week of hourly data, loosely modeled on a small boiler plant:

| Tag | Description | Normal range |
| --- | --- | --- |
| `FEEDWATER.FLOW_GPM` | Feedwater flow | ~460-540 gpm, mild diurnal swing |
| `BOILER.PRESSURE_PSI` | Drum pressure | ~150 psi, low noise |
| `STACK.TEMP_F` | Stack gas temperature | ~320 °F, low noise |

We generate clean signals first, then deliberately inject four kinds of problems
that show up constantly in real historian exports:

- **Range violation** — a stack temperature spike to 900 °F (sensor fault / bad read)
- **Flatline** — drum pressure frozen at exactly 150.0 psi for 6 hours (stuck sensor)
- **Nulls** — two missing flow readings
- **Timestamp gap** — 3 hours of pressure readings missing entirely (historian/collector outage)

(We also throw in a flow spike to 900 gpm, which the built-in *delta* / *range* rules
both catch — a nice example of two rules agreeing on one bad point.)


In [ ]:
rng = np.random.default_rng(7)

start = pd.Timestamp("2026-02-01 00:00:00", tz="UTC")
periods = 24 * 7  # one week, hourly
timestamps = pd.date_range(start, periods=periods, freq="1h")
t = np.arange(periods)

flow = 500 + 40 * np.sin(2 * np.pi * t / 24) + rng.normal(0, 5, periods)
pressure = 150 + rng.normal(0, 1.5, periods)
temp = 320 + rng.normal(0, 3, periods)

rows = []
for i, ts in enumerate(timestamps):
    rows.append((ts, "FEEDWATER.FLOW_GPM", flow[i]))
    rows.append((ts, "BOILER.PRESSURE_PSI", pressure[i]))
    rows.append((ts, "STACK.TEMP_F", temp[i]))

df = pd.DataFrame(rows, columns=["timestamp", "tag_name", "value"])
df.shape


In [ ]:
# --- inject anomalies -------------------------------------------------

# 1) Flatline: pressure stuck at exactly 150.0 psi for 6 hours on day 3
flat_mask = (
    (df.tag_name == "BOILER.PRESSURE_PSI")
    & (df.timestamp >= timestamps[48])
    & (df.timestamp < timestamps[54])
)
df.loc[flat_mask, "value"] = 150.0

# 2) Range violation: stack temp spikes to 900 F (way outside 250-450 F)
range_mask = (df.tag_name == "STACK.TEMP_F") & (df.timestamp == timestamps[80])
df.loc[range_mask, "value"] = 900.0

# 3) Nulls: two missing flow readings
null_mask = (df.tag_name == "FEEDWATER.FLOW_GPM") & (df.timestamp.isin(timestamps[100:102]))
df.loc[null_mask, "value"] = np.nan

# 4) Gap: drop 3 hours of pressure rows entirely (collector outage, not just a null)
gap_mask = (df.tag_name == "BOILER.PRESSURE_PSI") & (df.timestamp.isin(timestamps[120:123]))
df = df[~gap_mask]

# 5) Bonus: a flow spike caught by both the delta and range rules
spike_mask = (df.tag_name == "FEEDWATER.FLOW_GPM") & (df.timestamp == timestamps[140])
df.loc[spike_mask, "value"] = 900.0

df = df.sort_values(["tag_name", "timestamp"]).reset_index(drop=True)
import os
os.makedirs("data", exist_ok=True)  # gitignored, per CONTRIBUTING.md convention
df.to_csv("data/generated_boiler_scada.csv", index=False)
df.head()


## 2. Define quality rules in YAML

`default_rules` apply to every tag. `tag_rules` add tag-specific checks on top
(range limits are almost always tag-specific — 150 psi is fine for a drum, not for a
gearbox). This is the same config format a plant engineer could edit without touching
Python — see [`docs/yaml-configuration.md`](../docs/yaml-configuration.md) for the full
rule reference.


In [ ]:
yaml_config = """
default_rules:
  - check: null
    level: bad
  - check: flatline
    window: 3h
    min_delta: 0.01
    level: sus
  - check: delta
    max_delta: 150.0
    level: sus

tag_rules:
  "BOILER.PRESSURE_PSI":
    - check: range
      min: 100
      max: 250
      level: bad
    - check: flatline
      window: 4h
      min_delta: 0.01
      level: bad   # a stuck drum-pressure sensor is a bad-level problem, not just suspect

  "STACK.TEMP_F":
    - check: range
      min: 250
      max: 450
      level: bad

  "FEEDWATER.FLOW_GPM":
    - check: range
      min: 300
      max: 700
      level: bad
"""

with open("data/generated_tsqc_rules.yaml", "w") as f:
    f.write(yaml_config)

print(yaml_config)


## 3. Run the quality check

Five lines, same as the top-level README quickstart.


In [ ]:
result = tsqc.check(df, rules="data/generated_tsqc_rules.yaml")
result.df.head()


## 4. Inspect the results

### `summary()` — pct good/sus/bad per tag


In [ ]:
result.summary()

### `issue_summary()` — one row per contiguous bad/sus run, with the triggered rule(s)

In [ ]:
result.issue_summary()

### `check_timestamps()` — gaps, duplicates, non-monotonic timestamps

This is where the 3-hour pressure gap we dropped earlier shows up — it never appears
as a "bad" *value*, because there's no row to flag. It's a hole in the index, which is
exactly what this check is for.


In [ ]:
result.check_timestamps(expected_freq="1h")

## 5. Visualize: the quality timeline

One horizontal row per tag, colored green / yellow / red, with hover tooltips
showing the triggered rule and value. This is the chart that motivated the whole
library — nothing else in the open-source timeseries QC space renders this view
out of the box.


In [ ]:
fig = result.plot(title="Boiler Plant — Data Quality Timeline")
fig.show()


## 6. Export a standalone HTML report

Self-contained, no CDN dependency — safe to email or drop on a shared drive for
someone without a Python environment.


In [ ]:
result.export_report("data/generated_quickstart_report.html", title="Boiler Plant Quickstart — Data Quality Report")
print("wrote data/generated_quickstart_report.html")


## Next steps

- Swap in your own CSV (`timestamp`, `tag_name`, `value` columns) and point `assume_tz`
  at your historian's timezone if timestamps aren't already tz-aware.
- Full rule reference: [`docs/yaml-configuration.md`](../docs/yaml-configuration.md)
- Domain-specific end-to-end examples: [`examples/solar_farm.ipynb`](../examples/solar_farm.ipynb),
  [`examples/oilfield.ipynb`](../examples/oilfield.ipynb)
- Already have a historian quality/status column? See **External Quality Column** in the
  main [README](../README.md) for `exclusive` / `combined` / `none` modes.
